# 📘 Performance Optimization & Administration
## Databricks Notebook — Senior-Level Performance Mastery

**What you'll learn:**
- Spark execution model: jobs, stages, tasks, Catalyst, Tungsten
- AQE (Adaptive Query Execution)
- Shuffle optimization and data skew solutions
- Partitioning, Liquid Clustering, Predictive Optimization
- Caching, join optimization, Delta Lake tuning
- Query patterns, monitoring, Unity Catalog concepts

**Prerequisites:** Notebooks 01-06 (all datasets)

---

---
# 📗 Section 1: Spark Execution Model Deep Dive

## Jobs → Stages → Tasks

```
APPLICATION
└── JOB (triggered by each action: count, show, write)
    └── STAGE (separated by shuffle boundaries)
        └── TASK (one per partition, runs on one core)

Example: df.filter(...).groupBy(...).count()
         ├── Stage 1: filter (narrow, no shuffle)
         │   └── 200 tasks (one per partition)
         └── Stage 2: groupBy + count (wide, shuffle)
             └── 200 tasks (shuffle partitions)
```

## Catalyst Optimizer

Catalyst automatically optimizes your queries:
1. **Predicate pushdown** — filters pushed to data source
2. **Projection pruning** — only reads needed columns
3. **Join reordering** — smallest tables first
4. **Constant folding** — pre-computes constant expressions

In [ ]:
# Reading explain plans
spark.sql("USE techmart_dw")
df = spark.table("orders")

# Simple explain
print("=== SIMPLE PLAN ===")
df.filter("status = 'completed'").groupBy("payment_method").count().explain()

In [ ]:
# Extended explain (shows all optimization stages)
print("=== EXTENDED PLAN ===")
df.filter("status = 'completed' AND total_amount > 1000").select("order_id", "total_amount", "payment_method").explain(True)

In [ ]:
# ============================================
# 🎯 YOUR TURN: Read an Explain Plan
# ============================================
# Run this query and interpret the explain plan:
# - Join orders with customers
# - Filter completed orders > $2000
# - Group by customer_segment
# - Count and sum total_amount
#
# Questions to answer:
# 1. How many stages?
# 2. Where is the shuffle?
# 3. Is there a broadcast join or sort-merge join?
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *

orders = spark.table("orders")
customers = spark.table("customers")

result = (orders
    .join(customers, "customer_id")
    .filter("status = 'completed' AND total_amount > 2000")
    .groupBy("customer_segment")
    .agg(count("*").alias("cnt"), sum("total_amount").alias("revenue"))
)
result.explain(True)
# Stages: 3 (scan orders, scan customers, shuffle+aggregate)
# Shuffle: at the join and at the groupBy
# Join type: depends on table size — likely SortMergeJoin

---
# 📗 Section 2: Adaptive Query Execution (AQE)

**What AQE does automatically (enabled by default since Spark 3.2):**
1. **Dynamic partition coalescing** — merges small shuffle partitions
2. **Dynamic join switching** — switches to broadcast if one side is small
3. **Skew join optimization** — splits skewed partitions

In [ ]:
# Check AQE status
print("AQE Configuration:")
print(f"  spark.sql.adaptive.enabled = {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"  spark.sql.adaptive.coalescePartitions.enabled = {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled', 'not set')}")
print(f"  spark.sql.adaptive.skewJoin.enabled = {spark.conf.get('spark.sql.adaptive.skewJoin.enabled', 'not set')}")

In [ ]:
# AQE in action: dynamic coalescing
# Without AQE: 200 shuffle partitions (many empty/tiny)
# With AQE: automatically merges small partitions

from pyspark.sql.functions import *
import time

orders = spark.table("orders")

# This groupBy creates 200 shuffle partitions by default
# AQE will coalesce them to fewer, larger partitions
start = time.time()
result = orders.groupBy("status", "payment_method").agg(count("*"), sum("total_amount"))
result.collect()
print(f"Query with AQE: {time.time()-start:.2f}s")
print(f"Result partitions: {result.rdd.getNumPartitions()}")

---
# 📗 Section 3: Shuffle Optimization

**Shuffles are expensive** — data moves across the network between executors.

**What causes shuffles:**
- `groupBy()` / `agg()`
- `join()` (except broadcast)
- `distinct()` / `dropDuplicates()`
- `orderBy()` / `sort()`
- `repartition()`

In [ ]:
# Measure shuffle impact
from pyspark.sql.functions import *
import time

orders = spark.table("orders")

# BAD: Multiple shuffles
start = time.time()
bad = (orders
    .groupBy("customer_id").agg(count("*").alias("cnt"))
    .filter("cnt > 3")
    .orderBy("cnt", ascending=False)
    .limit(100)
)
bad.collect()
bad_time = time.time() - start

# BETTER: Reduce data before shuffle
start = time.time()
good = (orders
    .filter("status = 'completed'")  # Filter BEFORE groupBy (less data to shuffle)
    .groupBy("customer_id").agg(count("*").alias("cnt"))
    .filter("cnt > 3")
    .orderBy("cnt", ascending=False)
    .limit(100)
)
good.collect()
good_time = time.time() - start

print(f"Without pre-filter: {bad_time:.2f}s")
print(f"With pre-filter:    {good_time:.2f}s")
print(f"Improvement: {(1 - good_time/bad_time)*100:.0f}% faster")

## Shuffle Optimization — Detailed

### What Causes Shuffles

Every time Spark needs to group data by a key (for GROUP BY, JOIN, DISTINCT),
it must redistribute data across executors — this is a **shuffle**.

```
BEFORE SHUFFLE:                          AFTER SHUFFLE:
Executor 1: [A=100, B=200, C=300]       Executor 1: [A=100, A=150, A=200]
Executor 2: [A=150, C=400, D=500]  →    Executor 2: [B=200, B=300]
Executor 3: [B=300, D=600, A=200]       Executor 3: [C=300, C=400]
                                         Executor 4: [D=500, D=600]

Network I/O: ALL data moves across the network!
```

### Measuring Shuffle Cost

```python
# Check shuffle read/write in Spark UI
# Or use explain() to see shuffle stages:
df.groupBy("customer_id").agg(sum("amount")).explain()

# Look for "Exchange" in the plan — that's a shuffle:
# == Physical Plan ==
# *(2) HashAggregate(...)
# +- Exchange hashpartitioning(customer_id#0, 200)  ← SHUFFLE HERE
#    +- *(1) HashAggregate(...)
#       +- *(1) FileScan ...
```

### Reducing Shuffle Cost

| Technique | How | When |
|-----------|-----|------|
| **Broadcast join** | Send small table to all executors | One table < 10MB |
| **Reduce shuffle partitions** | `spark.sql.shuffle.partitions = N` | Small data |
| **Pre-partition data** | Partition by join key before writing | Repeated joins on same key |
| **AQE** | Auto-optimize at runtime | Always enable |
| **Avoid unnecessary shuffles** | Filter before groupBy/join | Always |

### The Shuffle Partition Problem

```
Default: spark.sql.shuffle.partitions = 200

For 1 GB of data:
  200 partitions × 5 MB each = OK

For 10 MB of data:
  200 partitions × 50 KB each = TERRIBLE (200 tiny tasks!)
  Fix: spark.conf.set("spark.sql.shuffle.partitions", "8")

For 1 TB of data:
  200 partitions × 5 GB each = TERRIBLE (OOM!)
  Fix: spark.conf.set("spark.sql.shuffle.partitions", "2000")
```

In [ ]:
# Shuffle partition tuning demonstration
from pyspark.sql.functions import col, count, sum as spark_sum
import time

print("⚙️ SHUFFLE PARTITION TUNING")
print("=" * 60)

orders = spark.table("techmart_dw.orders")
row_count = orders.count()
print(f"\nDataset: {row_count:,} rows")

# Check current setting
current = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Current shuffle.partitions: {current}")

# Demonstrate the impact
configs_to_test = [
    ("Too few (4)", "4"),
    ("Default (200)", "200"),
    ("Tuned (20)", "20"),
]

print("\n📊 Impact of shuffle.partitions on a GROUP BY query:")
print(f"{'Config':<20} {'Partitions':<12} {'Recommendation'}")
print("-" * 60)

for name, value in configs_to_test:
    spark.conf.set("spark.sql.shuffle.partitions", value)
    
    # Run a shuffle-heavy query
    start = time.perf_counter()
    result = (orders
        .groupBy("payment_method", "status")
        .agg(count("*").alias("cnt"), spark_sum("total_amount").alias("total"))
        .collect())
    elapsed = time.perf_counter() - start
    
    rec = ""
    if value == "4":
        rec = "⚠️ Too few — tasks are too large"
    elif value == "200":
        rec = "⚠️ Default — may be too many for small data"
    else:
        rec = "✅ Tuned — matches data size"
    
    print(f"{name:<20} {value:<12} {rec}")

# Reset to AQE-managed
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("\n✅ Reset to AQE-managed (AQE will auto-tune partitions)")
print("\n💡 Rule of thumb: shuffle.partitions ≈ 2-4x number of cores")
print(f"   Your cluster has {spark.sparkContext.defaultParallelism} cores")
print(f"   Recommended: {spark.sparkContext.defaultParallelism * 2}-{spark.sparkContext.defaultParallelism * 4} partitions")


---
# 📗 Section 4: Data Skew

**Problem:** When one partition has much more data than others, one task takes
forever while others finish quickly.

**Detection:** In Spark UI, look for tasks with much higher duration/data than median.

**Our data has skew:** The first 100 customers have disproportionately more orders.

In [ ]:
# Detect skew in orders table
from pyspark.sql.functions import *

orders = spark.table("orders")

# Check distribution
skew_check = (orders
    .groupBy("customer_id")
    .count()
    .orderBy("count", ascending=False)
)

print("Top 10 customers by order count (potential skew):")
skew_check.show(10)

# Statistics
stats = skew_check.agg(
    avg("count").alias("avg_orders"),
    max("count").alias("max_orders"),
    min("count").alias("min_orders"),
    percentile_approx("count", 0.5).alias("median_orders"),
    percentile_approx("count", 0.99).alias("p99_orders")
).collect()[0]

print(f"\nSkew metrics:")
print(f"  Average: {stats.avg_orders:.1f} orders/customer")
print(f"  Median:  {stats.median_orders}")
print(f"  P99:     {stats.p99_orders}")
print(f"  Max:     {stats.max_orders}")
print(f"  Skew ratio (max/median): {stats.max_orders/max(stats.median_orders,1):.1f}x")

In [ ]:
# Salting technique to fix skew in joins
from pyspark.sql.functions import *
import time

orders = spark.table("orders")
customers = spark.table("customers")

# Regular join (skewed — customer_id 1-100 have many orders)
start = time.time()
regular = orders.join(customers, "customer_id")
regular.count()
regular_time = time.time() - start

# Salted join: add random salt to break up hot keys
SALT_BUCKETS = 10

# Salt the large table (orders)
orders_salted = orders.withColumn("_salt", (rand() * SALT_BUCKETS).cast("int"))
orders_salted = orders_salted.withColumn("_join_key", 
    concat(col("customer_id").cast("string"), lit("_"), col("_salt").cast("string")))

# Explode the small table (customers) to match all salt values
from pyspark.sql.functions import explode, array, lit, sequence
customers_exploded = (customers
    .withColumn("_salt", explode(sequence(lit(0), lit(SALT_BUCKETS - 1))))
    .withColumn("_join_key",
        concat(col("customer_id").cast("string"), lit("_"), col("_salt").cast("string")))
)

start = time.time()
salted = orders_salted.join(customers_exploded, "_join_key")
salted.count()
salted_time = time.time() - start

print(f"Regular join: {regular_time:.2f}s")
print(f"Salted join:  {salted_time:.2f}s")
print("Note: Salting helps most on very large skewed datasets")

## Data Skew — Causes and Solutions

### What is Data Skew?

Data skew occurs when data is unevenly distributed across partitions:

```
BALANCED (no skew):              SKEWED:
Partition 1: 1M rows  ✅         Partition 1: 10M rows  ← SLOW!
Partition 2: 1M rows  ✅         Partition 2: 100K rows ✅
Partition 3: 1M rows  ✅         Partition 3: 50K rows  ✅
Partition 4: 1M rows  ✅         Partition 4: 200K rows ✅

All tasks finish ~same time      Task 1 takes 100x longer!
```

### Common Causes of Skew

1. **NULL values in join key** — all NULLs go to one partition
2. **Popular keys** — "Amazon" customer has 10M orders, others have 100
3. **Uneven partitioning** — data partitioned by a skewed column
4. **Hot dates** — Black Friday has 100x normal order volume

### Solutions

#### Solution 1: AQE Skew Join (Automatic)
```python
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
# AQE automatically splits skewed partitions
```

#### Solution 2: Salting (Manual)
```python
# Add random prefix to distribute hot keys
from pyspark.sql.functions import concat, lit, floor, rand

# Salt the skewed table
salted_orders = orders.withColumn(
    "salted_customer_id",
    concat(col("customer_id"), lit("_"), (floor(rand() * 10)).cast("string"))
)

# Salt the dimension table (replicate 10x)
from pyspark.sql.functions import explode, array
salted_customers = customers.withColumn(
    "salt", explode(array([lit(str(i)) for i in range(10)]))
).withColumn(
    "salted_customer_id",
    concat(col("customer_id"), lit("_"), col("salt"))
)

# Join on salted key (no skew!)
result = salted_orders.join(salted_customers, "salted_customer_id")
```

#### Solution 3: Filter NULLs Before Join
```python
# Handle NULLs separately
non_null = orders.filter(col("customer_id").isNotNull())
null_orders = orders.filter(col("customer_id").isNull())

result = non_null.join(customers, "customer_id")
# Handle null_orders separately (e.g., assign to "Unknown" customer)
```

In [ ]:
# Data skew detection and handling
from pyspark.sql.functions import col, count, sum as spark_sum, max as spark_max, min as spark_min

print("🔍 DATA SKEW DETECTION")
print("=" * 60)

orders = spark.table("techmart_dw.orders")

# Step 1: Detect skew — check distribution of data by key
print("\n📊 Order distribution by customer (checking for skew):")
customer_order_counts = (orders
    .groupBy("customer_id")
    .agg(count("*").alias("order_count"))
    .orderBy(col("order_count").desc()))

# Show distribution statistics
stats = customer_order_counts.agg(
    spark_max("order_count").alias("max_orders"),
    spark_min("order_count").alias("min_orders"),
    (spark_max("order_count") / spark_min("order_count")).alias("skew_ratio")
).collect()[0]

print(f"   Max orders per customer: {stats['max_orders']}")
print(f"   Min orders per customer: {stats['min_orders']}")
print(f"   Skew ratio: {stats['skew_ratio']:.1f}x")

if stats['skew_ratio'] > 10:
    print("   ⚠️ SKEW DETECTED! Some customers have much more data than others.")
else:
    print("   ✅ Data is relatively balanced.")

# Show top 5 customers by order count
print("\n   Top 5 customers by order count:")
customer_order_counts.show(5)

# Step 2: Check if AQE skew join is enabled
aqe_skew = spark.conf.get("spark.sql.adaptive.skewJoin.enabled", "false")
print(f"\n⚙️ AQE Skew Join enabled: {aqe_skew}")
if aqe_skew.lower() != "true":
    print("   💡 Enable with: spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    print("   ✅ Enabled AQE skew join handling")


---
# 📗 Section 5: Partitioning Strategies

## Choosing Partition Columns

Good partition columns have:
- Low cardinality (< 1000 distinct values)
- Frequently used in WHERE clauses
- Even distribution of data

**Over-partitioning:** Too many small files → slow reads
**Under-partitioning:** Files too large → no pruning benefit

In [ ]:
%sql
-- Create a partitioned version of website_events
USE techmart_dw;

DROP TABLE IF EXISTS website_events_partitioned;
CREATE TABLE website_events_partitioned
PARTITIONED BY (event_date)
AS SELECT * FROM website_events;

-- Verify partition pruning
EXPLAIN SELECT COUNT(*) FROM website_events_partitioned WHERE event_date = '2024-03-15';

In [ ]:
%sql
-- Compare: with vs without partition pruning
-- WITH pruning (only reads one partition):
SELECT COUNT(*) AS with_pruning FROM website_events_partitioned WHERE event_date = '2024-03-15';

In [ ]:
%sql
-- Show partitions
SHOW PARTITIONS website_events_partitioned LIMIT 10;

---
# 📗 Section 6: Caching & Persistence

## When to Cache
- DataFrame used **multiple times** in the same job
- Expensive computation (joins, aggregations) reused downstream
- Iterative algorithms

## When NOT to Cache
- One-time reads
- Very large DataFrames (won't fit in memory)
- Data that changes between uses

In [ ]:
import time
from pyspark.sql.functions import *

# Demo: caching benefit for repeated access
orders = spark.table("orders").filter("status = 'completed'")

# Without cache: reads from disk each time
start = time.time()
orders.count()
orders.groupBy("payment_method").count().collect()
orders.agg(sum("total_amount")).collect()
no_cache_time = time.time() - start

# With cache: first call loads, subsequent calls use memory
orders.cache()
start = time.time()
orders.count()  # triggers cache fill
orders.groupBy("payment_method").count().collect()
orders.agg(sum("total_amount")).collect()
cache_time = time.time() - start

orders.unpersist()

print(f"Without cache (3 actions): {no_cache_time:.2f}s")
print(f"With cache (3 actions):    {cache_time:.2f}s")

---
# 📗 Section 7: Join Optimization

## Join Types by Performance

| Join Type | When Used | Performance |
|---|---|---|
| Broadcast Hash | Small table (< 10MB) | ⚡ Fastest |
| Sort Merge | Large + Large | 🔄 Default |
| Shuffle Hash | Medium tables | 🔄 Sometimes faster |
| Broadcast Nested Loop | Cross join / non-equi | 🐌 Slowest |

In [ ]:
from pyspark.sql.functions import broadcast
import time

orders = spark.table("orders")
customers = spark.table("customers")

# Default join (Sort Merge — shuffles both sides)
start = time.time()
default_join = orders.join(customers, "customer_id")
default_join.count()
default_time = time.time() - start

# Broadcast join (sends small table to all executors — no shuffle)
start = time.time()
broadcast_join = orders.join(broadcast(customers), "customer_id")
broadcast_join.count()
broadcast_time = time.time() - start

print(f"Sort Merge Join: {default_time:.2f}s")
print(f"Broadcast Join:  {broadcast_time:.2f}s")
print(f"\nBroadcast is faster because customers table is small enough to broadcast")

In [ ]:
%sql
-- SQL hint for broadcast
SELECT /*+ BROADCAST(c) */
  o.order_id, c.first_name, o.total_amount
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
LIMIT 10;

---
# 📗 Section 8: Delta Lake Optimization

## 8.1 OPTIMIZE (File Compaction)

Small files = slow reads. OPTIMIZE merges them into larger files (target ~1GB).

In [ ]:
%sql
-- Check file metrics before optimization
DESCRIBE DETAIL orders;

In [ ]:
%sql
-- OPTIMIZE: compact small files
OPTIMIZE orders;

## 8.2 ZORDER

Z-ordering co-locates related data in the same files, enabling data skipping.
Choose columns that are frequently used in WHERE clauses.

In [ ]:
%sql
-- ZORDER by frequently filtered columns
OPTIMIZE orders ZORDER BY (customer_id, order_date);

## 8.3 Liquid Clustering (RECOMMENDED for new tables — GA since May 2024)

**Replaces both partitioning AND ZORDER** for new tables.

Key advantages:
- **Incremental:** Only re-clusters new/modified data (not full rewrite like ZORDER)
- **Changeable:** `ALTER TABLE ... CLUSTER BY` without rewriting data
- **Auto-maintained:** Works with Predictive Optimization

Reference: https://docs.databricks.com/delta/clustering.html

In [ ]:
%sql
-- Liquid Clustering: create a new table with clustering
DROP TABLE IF EXISTS orders_clustered;
CREATE TABLE orders_clustered
CLUSTER BY (customer_id, order_date)
AS SELECT * FROM orders;

-- Verify clustering
DESCRIBE DETAIL orders_clustered;

In [ ]:
%sql
-- Change clustering keys without rewriting (unique to Liquid Clustering!)
ALTER TABLE orders_clustered CLUSTER BY (status, payment_method);

-- Trigger clustering optimization
OPTIMIZE orders_clustered;

## 8.4 Predictive Optimization (Default since Nov 2024)

**AI-driven automatic maintenance** for Unity Catalog managed tables.
Automatically runs OPTIMIZE, VACUUM, and ANALYZE based on access patterns.

- No manual scheduling needed
- Runs during low-usage periods
- No extra charge for operations
- View history: `system.storage.predictive_optimization_operations_history`

Reference: https://docs.databricks.com/aws/en/optimizations/predictive-optimization

⚠️ Requires Unity Catalog (not available on Community Edition)

## 8.5 VACUUM

Removes old file versions no longer needed for time travel.
**Default retention: 7 days.** Cannot undo after VACUUM.

In [ ]:
%sql
-- Check history before vacuum
DESCRIBE HISTORY orders LIMIT 5;

In [ ]:
%sql
-- VACUUM with default retention (7 days)
-- Only removes files older than retention period
VACUUM orders;

## 8.6 Table Statistics

In [ ]:
%sql
-- Collect statistics for better query planning
ANALYZE TABLE orders COMPUTE STATISTICS FOR ALL COLUMNS;

In [ ]:
%sql
-- View column statistics
DESCRIBE EXTENDED orders total_amount;

## Delta Lake Optimization — Before & After Benchmarks

### The Small Files Problem

After many incremental writes, Delta tables accumulate thousands of small files:

```
BEFORE OPTIMIZE:                         AFTER OPTIMIZE:
├── part-001.parquet (2 MB)              ├── part-001.parquet (128 MB)
├── part-002.parquet (500 KB)            ├── part-002.parquet (128 MB)
├── part-003.parquet (1 MB)              └── part-003.parquet (95 MB)
├── ... (500 more tiny files)
└── part-500.parquet (1 MB)

Query time: 45 seconds                   Query time: 3 seconds (15x faster!)
Files read: 500                          Files read: 3
```

### Liquid Clustering vs ZORDER vs Partitioning

| Aspect | Partitioning | ZORDER | Liquid Clustering |
|--------|-------------|--------|-------------------|
| **Setup** | At table creation | Run OPTIMIZE | At creation or ALTER |
| **Change keys** | ❌ Must rewrite | ❌ Must re-ZORDER | ✅ ALTER TABLE |
| **Incremental** | N/A | ❌ Full rewrite | ✅ Only new files |
| **Auto-maintenance** | ❌ Manual | ❌ Manual | ✅ Predictive Optimization |
| **Best for** | Low-cardinality (date, region) | High-cardinality (customer_id) | Everything (2024+) |

### Choosing Clustering Keys

**Good clustering keys:**
- Columns frequently used in WHERE clauses
- Columns used in JOIN conditions
- High-cardinality columns (dates, IDs)

**Bad clustering keys:**
- Boolean columns (only 2 values)
- Columns rarely filtered on
- Columns with extreme skew

In [ ]:
# Delta optimization demonstration
from pyspark.sql.functions import col, count

print("⚡ DELTA LAKE OPTIMIZATION DEMO")
print("=" * 60)

# Check current table state
print("\n📊 Table details before optimization:")
spark.sql("DESCRIBE DETAIL techmart_dw.orders").select(
    "numFiles", "sizeInBytes", "numRows"
).show()

# Run OPTIMIZE
print("\n🔄 Running OPTIMIZE...")
try:
    result = spark.sql("OPTIMIZE techmart_dw.orders")
    result.show(truncate=False)
    print("\n📊 Table details after OPTIMIZE:")
    spark.sql("DESCRIBE DETAIL techmart_dw.orders").select(
        "numFiles", "sizeInBytes", "numRows"
    ).show()
except Exception as e:
    print(f"   ⚠️ OPTIMIZE: {e}")
    print("   (OPTIMIZE requires Delta table with write permissions)")

# Show VACUUM syntax
print("\n🧹 VACUUM (remove old file versions):")
print("   VACUUM techmart_dw.orders RETAIN 168 HOURS  -- 7 days (default)")
print("   VACUUM techmart_dw.orders DRY RUN            -- Preview only")
print("   ⚠️ After VACUUM, time travel to old versions is impossible!")

# Show Liquid Clustering syntax
print("\n🧊 LIQUID CLUSTERING (recommended for new tables):")
print("""
   -- Create with clustering:
   CREATE TABLE gold.daily_revenue
   CLUSTER BY (order_date, region)
   AS SELECT ...
   
   -- Change clustering keys (no rewrite needed!):
   ALTER TABLE gold.daily_revenue CLUSTER BY (region, product_category)
   
   -- Trigger clustering:
   OPTIMIZE gold.daily_revenue
""")

---
# 📗 Section 9: Query Optimization Patterns

In [ ]:
# Pattern 1: Filter early (predicate pushdown)
from pyspark.sql.functions import *
import time

orders = spark.table("orders")
customers = spark.table("customers")

# BAD: filter after join (processes all rows through join)
start = time.time()
bad = orders.join(customers, "customer_id").filter("status = 'completed' AND customer_segment = 'Premium'")
bad.count()
bad_time = time.time() - start

# GOOD: filter before join (less data to shuffle)
start = time.time()
good = (orders.filter("status = 'completed'")
    .join(customers.filter("customer_segment = 'Premium'"), "customer_id"))
good.count()
good_time = time.time() - start

print(f"Filter AFTER join:  {bad_time:.2f}s")
print(f"Filter BEFORE join: {good_time:.2f}s")

In [ ]:
# Pattern 2: Select only needed columns (projection pushdown)
import time

# BAD: select * then filter columns later
start = time.time()
bad = spark.table("website_events").groupBy("event_type").count()
bad.collect()
bad_time = time.time() - start

# GOOD: select only needed columns first
start = time.time()
good = spark.table("website_events").select("event_type").groupBy("event_type").count()
good.collect()
good_time = time.time() - start

print(f"SELECT * then group: {bad_time:.2f}s")
print(f"SELECT col then group: {good_time:.2f}s")

In [ ]:
# Pattern 3: Use approximate functions for exploration
import time

events = spark.table("website_events")

# Exact distinct count (expensive)
start = time.time()
exact = events.select(countDistinct("session_id")).collect()[0][0]
exact_time = time.time() - start

# Approximate (much faster on large data)
start = time.time()
approx = events.select(approx_count_distinct("session_id")).collect()[0][0]
approx_time = time.time() - start

print(f"Exact count_distinct:  {exact:,} ({exact_time:.2f}s)")
print(f"Approx count_distinct: {approx:,} ({approx_time:.2f}s)")
print(f"Error: {abs(exact-approx)/exact*100:.1f}%")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Optimize Slow Queries
# ============================================
# Rewrite these slow queries to be fast:
#
# Query 1 (slow): 
#   spark.table("orders").join(spark.table("order_items"), "order_id")
#   .join(spark.table("products"), "product_id")
#   .filter("category = 'Electronics'")
#   .groupBy("category").agg(sum("line_total"))
#
# Query 2 (slow):
#   spark.table("website_events").distinct().count()
#
# Query 3 (slow):
#   spark.table("orders").collect()  # then process in Python
#
# Write optimized versions below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *

# Query 1: Filter products FIRST, then join (less data to shuffle)
products_elec = spark.table("products").filter("category = 'Electronics'").select("product_id", "category")
order_items = spark.table("order_items").select("order_id", "product_id", "line_total")
result1 = (order_items
    .join(broadcast(products_elec), "product_id")  # broadcast small filtered table
    .groupBy("category").agg(round(sum("line_total"), 2).alias("revenue"))
)
result1.show()

# Query 2: Don't use distinct() on full table — use approx or specific columns
approx_sessions = spark.table("website_events").select(approx_count_distinct("event_id")).collect()[0][0]
print(f"Approx distinct events: {approx_sessions:,}")

# Query 3: NEVER collect() large tables — use Spark operations instead
result3 = spark.table("orders").groupBy("status").agg(count("*"), sum("total_amount"))
result3.show()  # stays in Spark, never comes to driver

---
# 📗 Section 10: Cluster Sizing & Cost (Conceptual)

| Cluster Type | Use Case | Cost |
|---|---|---|
| All-Purpose | Development, exploration | Higher (always on) |
| Job Cluster | Scheduled pipelines | Lower (ephemeral) |
| SQL Warehouse | BI queries, dashboards | Pay per query |
| Serverless | Any (no cluster management) | Premium pricing |

**Cost optimization tips:**
- Use job clusters for production (auto-terminate)
- Spot instances for fault-tolerant workloads (70% savings)
- Right-size: start small, scale up based on metrics
- Autoscaling: set min/max workers based on workload
- Cluster pools: pre-warm instances for faster startup

---
# 📗 Section 11: Monitoring & Observability

In [ ]:
%sql
-- Table metrics
DESCRIBE DETAIL techmart_dw.orders;

In [ ]:
%sql
-- Table history (operations performed)
DESCRIBE HISTORY techmart_dw.orders LIMIT 10;

In [ ]:
# DataFrame-level metrics
from pyspark.sql.functions import *

orders = spark.table("techmart_dw.orders")
print(f"Partitions: {orders.rdd.getNumPartitions()}")
print(f"Schema: {len(orders.columns)} columns")
print(f"Rows: {orders.count():,}")

# Check for potential issues
null_counts = {}
for c in ["order_id", "customer_id", "total_amount"]:
    null_counts[c] = orders.filter(f"{c} IS NULL").count()
print(f"NULL counts: {null_counts}")

## Query Optimization Patterns — Practical Examples

### Pattern 1: Predicate Pushdown

Push filters as close to the data source as possible:

```python
# ❌ BAD: Filter after expensive join
result = (orders
    .join(customers, "customer_id")
    .join(products, "product_id")
    .filter(col("order_date") >= "2024-01-01")  # Filter AFTER joining!
    .filter(col("status") == "completed"))

# ✅ GOOD: Filter BEFORE joining
filtered_orders = orders.filter(
    (col("order_date") >= "2024-01-01") & (col("status") == "completed")
)
result = (filtered_orders
    .join(customers, "customer_id")
    .join(products, "product_id"))
```

### Pattern 2: Column Pruning

Read only the columns you need:

```python
# ❌ BAD: Read all 20 columns, use 3
df = spark.table("orders")  # Reads ALL columns
result = df.select("order_id", "customer_id", "total_amount")

# ✅ GOOD: Read only needed columns
df = spark.table("orders").select("order_id", "customer_id", "total_amount")
```

### Pattern 3: Avoid UDFs When Possible

Python UDFs bypass Catalyst optimization:

```python
# ❌ BAD: Python UDF (slow, no optimization)
from pyspark.sql.functions import udf
@udf("string")
def categorize(amount):
    if amount > 1000: return "high"
    elif amount > 100: return "medium"
    return "low"
df.withColumn("category", categorize(col("amount")))

# ✅ GOOD: Built-in functions (Catalyst-optimized)
from pyspark.sql.functions import when
df.withColumn("category",
    when(col("amount") > 1000, "high")
    .when(col("amount") > 100, "medium")
    .otherwise("low"))
```

### Pattern 4: Avoid collect() on Large Data

```python
# ❌ DANGEROUS: Brings ALL data to driver (OOM risk!)
all_orders = df.collect()  # 20M rows → driver crashes!

# ✅ SAFE: Use show() for inspection
df.show(20)

# ✅ SAFE: Use take() for small samples
sample = df.take(100)

# ✅ SAFE: Write to table instead of collecting
df.write.format("delta").saveAsTable("gold.result")
```

In [ ]:
# Query optimization comparison
from pyspark.sql.functions import col, when, count, sum as spark_sum
import time

print("🚀 QUERY OPTIMIZATION PATTERNS — Before & After")
print("=" * 60)

orders = spark.table("techmart_dw.orders")
customers = spark.table("techmart_dw.customers")

# Pattern 1: Filter early vs late
print("\n1️⃣ FILTER EARLY vs LATE:")

# Late filter (bad)
start = time.perf_counter()
late_filter = (orders
    .join(customers, "customer_id")
    .filter(col("status") == "completed")
    .filter(col("customer_segment") == "gold")
    .count())
late_time = time.perf_counter() - start

# Early filter (good)
start = time.perf_counter()
early_filter = (orders
    .filter(col("status") == "completed")
    .join(customers.filter(col("customer_segment") == "gold"), "customer_id")
    .count())
early_time = time.perf_counter() - start

print(f"   Late filter:  {late_time:.4f}s  (result: {late_filter:,})")
print(f"   Early filter: {early_time:.4f}s  (result: {early_filter:,})")
speedup = late_time / max(early_time, 0.0001)
print(f"   Speedup: {speedup:.1f}x {'✅' if speedup > 1 else '(similar — AQE may have optimized)'}")

# Pattern 2: CASE WHEN vs UDF
print("\n2️⃣ CASE WHEN vs Python UDF:")

# Built-in CASE WHEN
start = time.perf_counter()
case_result = (orders
    .withColumn("value_tier",
        when(col("total_amount") > 500, "high")
        .when(col("total_amount") > 100, "medium")
        .otherwise("low"))
    .groupBy("value_tier").count()
    .collect())
case_time = time.perf_counter() - start

print(f"   CASE WHEN: {case_time:.4f}s")
print(f"   Result: {sorted([(r['value_tier'], r['count']) for r in case_result])}")
print(f"   ✅ CASE WHEN uses Catalyst optimization (vectorized execution)")
print(f"   ❌ Python UDF would be 5-10x slower (row-by-row Python execution)")


---
# 📗 Section 12: Unity Catalog Concepts (Conceptual)

## Three-Level Namespace

```
CATALOG (e.g., 'production')
└── SCHEMA (e.g., 'techmart_dw')
    └── TABLE (e.g., 'orders')

Full reference: production.techmart_dw.orders
```

## Key Features
- **Access Control:** GRANT/REVOKE at any level
- **Data Lineage:** Automatic tracking of data flow
- **Row-Level Security:** Filter rows based on user identity
- **Column Masking:** Hide sensitive columns (PII)
- **Audit Logging:** Who accessed what, when

⚠️ Unity Catalog requires Databricks Pro/Enterprise (not Community Edition)

---
# 🚀 Section 13: Mini Projects

## Project 1: Performance Audit

In [ ]:
# Performance audit of techmart_dw.orders
from pyspark.sql.functions import *
import time

spark.sql("USE techmart_dw")

def audit_table(table_name):
    """Run a performance audit on a table."""
    print(f"{'='*60}")
    print(f"PERFORMANCE AUDIT: {table_name}")
    print(f"{'='*60}")
    
    df = spark.table(table_name)
    
    # Basic metrics
    row_count = df.count()
    col_count = len(df.columns)
    partitions = df.rdd.getNumPartitions()
    
    print(f"  Rows: {row_count:,}")
    print(f"  Columns: {col_count}")
    print(f"  Partitions: {partitions}")
    print(f"  Avg rows/partition: {row_count//max(partitions,1):,}")
    
    # Skew check on key columns
    print(f"\n  Skew Analysis:")
    for col_name in ["customer_id", "status", "payment_method"]:
        stats = df.groupBy(col_name).count().agg(
            max("count").alias("max_count"),
            min("count").alias("min_count"),
            avg("count").alias("avg_count")
        ).collect()[0]
        skew_ratio = stats.max_count / max(stats.avg_count, 1)
        flag = "⚠️ SKEWED" if skew_ratio > 5 else "✅ OK"
        print(f"    {col_name}: max={stats.max_count}, avg={stats.avg_count:.0f}, ratio={skew_ratio:.1f}x {flag}")
    
    # Recommendations
    print(f"\n  Recommendations:")
    if partitions < 4:
        print(f"    ⚠️ Too few partitions ({partitions}) — consider repartition")
    if row_count / max(partitions, 1) > 1000000:
        print(f"    ⚠️ Large partitions — consider more partitions")
    print(f"    ✅ Run OPTIMIZE for file compaction")
    print(f"    ✅ Consider Liquid Clustering on (customer_id, order_date)")
    print(f"{'='*60}")

audit_table("orders")

## Project 2: Pipeline Optimization

In [ ]:
# Optimize this slow pipeline
from pyspark.sql.functions import *
import time

# SLOW VERSION (intentionally bad)
start = time.time()
orders = spark.table("orders")
items = spark.table("order_items")
products = spark.table("products")
customers = spark.table("customers")

# Bad: no filters before join, select * everywhere
slow = (orders
    .join(items, "order_id")
    .join(products, "product_id")
    .join(customers, "customer_id")
    .filter("status = 'completed' AND category = 'Electronics' AND customer_segment = 'Premium'")
    .groupBy("category", "customer_segment")
    .agg(count("*").alias("cnt"), sum("line_total").alias("revenue"))
)
slow.collect()
slow_time = time.time() - start

# FAST VERSION (optimized)
start = time.time()
# Filter early, select only needed columns, broadcast small tables
orders_f = orders.filter("status = 'completed'").select("order_id", "customer_id")
products_f = products.filter("category = 'Electronics'").select("product_id", "category")
customers_f = customers.filter("customer_segment = 'Premium'").select("customer_id", "customer_segment")
items_f = items.select("order_id", "product_id", "line_total")

fast = (orders_f
    .join(broadcast(customers_f), "customer_id")
    .join(items_f, "order_id")
    .join(broadcast(products_f), "product_id")
    .groupBy("category", "customer_segment")
    .agg(count("*").alias("cnt"), round(sum("line_total"), 2).alias("revenue"))
)
fast.collect()
fast_time = time.time() - start

print(f"Slow pipeline: {slow_time:.2f}s")
print(f"Fast pipeline: {fast_time:.2f}s")
print(f"Speedup: {slow_time/fast_time:.1f}x faster")
fast.show()

---
# 🏆 Section 14: Interview Questions

## Q1: How do you identify and fix data skew?

**Identify:**
- Spark UI: one task takes 10x+ longer than others
- `groupBy(key).count().orderBy(desc("count"))` — check distribution
- Skew ratio: max/median > 10x = significant skew

**Fix:**
1. **Broadcast join** — if one side is small (< 10MB)
2. **Salting** — add random suffix to hot keys, explode small table
3. **AQE skew handling** — automatic in Spark 3.2+
4. **Repartition** — redistribute data more evenly
5. **Isolate hot keys** — process separately, union results

## Q2: OPTIMIZE vs ZORDER vs Partitioning vs Liquid Clustering

| Feature | OPTIMIZE | ZORDER | Partitioning | Liquid Clustering |
|---|---|---|---|---|
| Purpose | Compact files | Co-locate data | Physical dirs | Smart layout |
| Incremental | No (full rewrite) | No | N/A | ✅ Yes |
| Changeable | N/A | Must re-ZORDER | Must rewrite | ✅ ALTER TABLE |
| Auto-maintained | With Pred. Opt. | No | No | ✅ With Pred. Opt. |
| Recommendation | Always useful | Legacy tables | Very large tables | **New tables** |

## Q3: Broadcast join vs Sort Merge join?

- **Broadcast:** One table < 10MB. Sends small table to all executors. No shuffle. ⚡ Fast.
- **Sort Merge:** Both tables large. Shuffles both, sorts, merges. Default. Reliable.
- **Tip:** Increase `spark.sql.autoBroadcastJoinThreshold` for larger dimension tables.

## Q4: How does AQE help?

1. **Dynamic coalescing** — merges tiny shuffle partitions → fewer tasks
2. **Dynamic join switching** — if runtime stats show one side is small, switches to broadcast
3. **Skew handling** — splits oversized partitions into smaller ones
4. All automatic, no code changes needed.

## Q5: Walk through optimizing a slow Spark job

1. **Check Spark UI** — find the slowest stage
2. **Read explain plan** — identify shuffles, join types
3. **Filter early** — push predicates before joins/shuffles
4. **Project early** — select only needed columns
5. **Broadcast small tables** — eliminate shuffles
6. **Check skew** — salt or isolate hot keys
7. **Cache if reused** — avoid recomputation
8. **OPTIMIZE + Liquid Clustering** — improve file layout
9. **Right-size cluster** — more cores for parallelism

---
# ✅ Notebook Complete!

**What was covered:**
- Spark execution model: jobs, stages, tasks, Catalyst, Tungsten
- AQE: dynamic coalescing, join switching, skew handling
- Shuffle optimization: filter early, reduce data volume
- Data skew: detection, salting technique, broadcast
- Partitioning: choosing columns, pruning verification
- Caching: when to use, when to avoid
- Join optimization: broadcast, sort merge, hints
- Delta Lake: OPTIMIZE, ZORDER, Liquid Clustering, Predictive Optimization, VACUUM
- Query patterns: filter pushdown, projection, approximate functions
- Monitoring: DESCRIBE DETAIL, HISTORY, statistics
- Unity Catalog concepts
- 2 mini projects: Performance Audit, Pipeline Optimization
- 5 senior-level interview questions

**Next:** Notebook 08 — Star Schema Modeling

---